<h1 align="center"><b> Cardiovascular Disease Prediction: A Complete EDA, Machine Learning, Deep Learning & NLP Pipeline</b></h1>

# The objective of this project is to analyze patient clinical data to identify key patterns associated with cardiovascular disease risk. Using features such as age, BMI, blood pressure, cholesterol, glucose levels, and gender, the study applies Exploratory Data Analysis (EDA), statistical techniques, and advanced modeling approaches to gain meaningful insights and build predictive systems for early detection and improved healthcare decision-making.
</p>

# 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'colab'
from scipy.stats import ttest_ind, chi2_contingency, pearsonr
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, classification_report)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.feature_extraction.text import TfidfVectorizer
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import re
import warnings
warnings.filterwarnings('ignore')
print(" All libraries imported.")

# 2. Load the Dataset

In [ ]:
df = pd.read_csv("/content/NACC_APOE_CVD_filtered (2).csv")

In [ ]:
df.head()

,NACCID,SEX,BIRTHYR,NACCAPOE,DEMENTED,CVHATT,HATTMULT,CVAFIB,CVANGIO,CVBYPASS,...,STROKE,STROKIF,STROKDEC,STKIMAG,CVD,CVDIF,VASC,VASCIF,VASCPS,VASCPSIF
0,NACC000011,2,1944,1.0,0,0.0,NaN,0.0,0.0,0.0,...,0.0,7.0,NaN,NaN,NaN,NaN,0.0,7.0,NaN,NaN
1,NACC000034,2,1935,4.0,0,0.0,8.0,0.0,0.0,0.0,...,NaN,NaN,8.0,8.0,0.0,7.0,NaN,NaN,NaN,NaN
2,NACC000067,1,1952,1.0,0,0.0,NaN,0.0,0.0,0.0,...,0.0,7.0,NaN,NaN,NaN,NaN,0.0,7.0,0.0,7.0
3,NACC000095,1,1926,2.0,1,0.0,NaN,0.0,0.0,0.0,...,0.0,7.0,NaN,NaN,NaN,NaN,0.0,7.0,0.0,7.0
4,NACC000144,1,1930,1.0,0,0.0,NaN,1.0,0.0,0.0,...,0.0,8.0,NaN,NaN,NaN,NaN,8.0,8.0,8.0,8.0


# 3. Dataset Overview & Cleaning

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40686 entries, 0 to 40685
Data columns (total 43 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   NACCID    40686 non-null  object 
 1   SEX       40686 non-null  int64  
 2   BIRTHYR   40686 non-null  int64  
 3   NACCAPOE  40686 non-null  float64
 4   DEMENTED  40686 non-null  int64  
 5   CVHATT    29582 non-null  float64
 6   HATTMULT  7713 non-null   float64
 7   CVAFIB    29536 non-null  float64
 8   CVANGIO   29624 non-null  float64
 9   CVBYPASS  29633 non-null  float64
 10  CVPACDEF  7742 non-null   float64
 11  CVPACE    21901 non-null  float64
 12  CVCHF     29598 non-null  float64
 13  CVANGINA  7738 non-null   float64
 14  CVHVALVE  7738 non-null   float64
 15  CVOTHR    29536 non-null  float64
 16  CVOTHRX   3347 non-null   object 
 17  MYOINF    18764 non-null  float64
 18  CONGHRT   18764 non-null  float64
 19  AFIBRILL  18764 non-null  float64
 20  ANGINA    18764 non-null  fl

In [ ]:
df.describe().T

,SEX,BIRTHYR,NACCAPOE,DEMENTED,CVHATT,HATTMULT,CVAFIB,CVANGIO,CVBYPASS,CVPACDEF,...,STROKE,STROKIF,STROKDEC,STKIMAG,CVD,CVDIF,VASC,VASCIF,VASCPS,VASCPSIF
count,40686.000000,40686.000000,40686.000000,40686.000000,29582.000000,7713.000000,29536.000000,29624.000000,29633.000000,7742.000000,...,21922.000000,21922.000000,18764.000000,18674.000000,18764.000000,18764.000000,21922.000000,21922.000000,15339.000000,15296.000000
mean,1.565969,1940.892912,1.819471,0.359878,0.110743,7.732141,0.100623,0.118823,0.079101,0.020408,...,0.041602,7.191406,7.852590,7.895041,0.076903,7.128011,2.801387,7.253353,2.945564,7.207178
std,0.495635,12.652103,1.061846,0.479970,0.446419,1.422104,0.365841,0.445972,0.379632,0.158624,...,0.199683,1.032187,1.050306,0.861236,0.266444,1.498858,3.800684,0.903614,3.832178,1.091093
min,1.000000,1896.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000
25%,1.000000,1932.000000,1.000000,0.000000,0.000000,8.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,7.000000,8.000000,8.000000,0.000000,7.000000,0.000000,7.000000,0.000000,7.000000
50%,2.000000,1941.000000,2.000000,0.000000,0.000000,8.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,7.000000,8.000000,8.000000,0.000000,7.000000,0.000000,7.000000,0.000000,7.000000
75%,2.000000,1949.000000,2.000000,1.000000,0.000000,8.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,8.000000,8.000000,8.000000,0.000000,8.000000,8.000000,8.000000,8.000000,8.000000
max,2.000000,2003.000000,6.000000,1.000000,2.000000,8.000000,2.000000,2.000000,2.000000,2.000000,...,1.000000,8.000000,8.000000,8.000000,1.000000,8.000000,8.000000,8.000000,8.000000,8.000000


In [ ]:
df.isnull().sum()

,0
NACCID,0
SEX,0
BIRTHYR,0
NACCAPOE,0
DEMENTED,0
CVHATT,11104
HATTMULT,32973
CVAFIB,11150
CVANGIO,11062
CVBYPASS,11053


# impute the missing value

In [ ]:
df.isnull().sum()

,0
NACCID,0
SEX,0
BIRTHYR,0
NACCAPOE,0
DEMENTED,0
CVHATT,0
HATTMULT,0
CVAFIB,0
CVANGIO,0
CVBYPASS,0


# 4. Exploratory Data Analysis (EDA)

# 4.1 Target Distribution

In [ ]:
fig = px.pie(df, names='target', title='Heart Disease Distribution (1=Disease, 0=No Disease)',
             color_discrete_sequence=['lightgreen','coral'])
fig.show()

# 4.2 Age Distribution by Target

In [ ]:
fig = px.histogram(df, x='age', color='target', barmode='overlay', opacity=0.6,
                   title='Age Distribution by Heart Disease Status')
fig.show()

# 4.3 Gender Distribution

In [ ]:
# Map sex: 1=male, 0=female
fig = px.histogram(df, x='sex', color='target', barmode='group',
                   title='Gender Distribution by Heart Disease (1=Male, 0=Female)')
fig.show()

# 4.4 Chest Pain Type Analysis

In [ ]:
fig = px.histogram(df, x='cp', color='target', barmode='group',
                   title='Chest Pain Type Distribution (cp: 1-4)')
fig.show()

# 4.5 Resting Blood Pressure vs Target

In [ ]:
fig = px.box(df, x='target', y='trestbps', color='target',
             title='Resting Blood Pressure (trestbps) by Heart Disease Status')
fig.show()

# 4.6 Cholesterol Levels by Target

In [ ]:
fig = px.box(df, x='target', y='chol', color='target',
             title='Serum Cholesterol (chol) by Heart Disease Status')
fig.show()

# 4.7 Maximum Heart Rate Analysis

In [ ]:
fig = px.violin(df, x='target', y='thalach', box=True,
                title='Maximum Heart Rate (thalach) by Heart Disease Status')
fig.show()

# 4.8 Exercise-Induced Angina

In [ ]:
fig = px.histogram(df, x='exang', color='target', barmode='group',
                   title='Exercise-Induced Angina (exang) by Heart Disease')
fig.show()

# 4.9 ST Depression (Oldpeak) Analysis

In [ ]:
fig = px.box(df, x='target', y='oldpeak', color='target',
             title='ST Depression (oldpeak) by Heart Disease Status')
fig.show()

# 4.10 Number of Major Vessels

In [ ]:
fig = px.histogram(df, x='ca', color='target', barmode='group',
                   title='Number of Major Vessels Colored (ca)')
fig.show()

# 4.11 Thalassemia Type Analysis

In [ ]:
fig = px.histogram(df, x='thal', color='target', barmode='group',
                   title='Thalassemia Type (thal) Distribution')
fig.show()

# 4.12 Correlation Heatmap

In [ ]:
# Select numerical columns
num_cols = df.select_dtypes(include=np.number).columns
corr = df[num_cols].corr()
fig = px.imshow(corr, text_auto=True, title='Correlation Heatmap',
                color_continuous_scale='RdBu', zmin=-1, zmax=1)
fig.show()

# 4.13 Pairplot of Key Features (using Plotly Scatter Matrix)

In [ ]:
key_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'target']
fig = px.scatter_matrix(df[key_features], dimensions=key_features[:-1], color='target',
                        title='Pairplot of Key Clinical Features')
fig.update_traces(diagonal_visible=False)
fig.show()

# 4.14 3D Scatter Plot

In [ ]:
fig = px.scatter_3d(df, x='age', y='chol', z='thalach', color='target',
                    title='3D View: Age, Cholesterol, Max Heart Rate')
fig.show()

# 5. Statistical Testing

# 5.1 T‑test for Continuous Features

In [ ]:
continuous_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
ttest_results = []
for col in continuous_features:
    group1 = df[df['target']==1][col]
    group0 = df[df['target']==0][col]
    t_stat, p_val = ttest_ind(group1, group0, equal_var=False)
    ttest_results.append({'Feature': col, 'T-statistic': t_stat, 'P-value': p_val})
ttest_df = pd.DataFrame(ttest_results).sort_values('P-value')
ttest_df['Significant'] = ttest_df['P-value'] < 0.05
print(ttest_df)

# 5.2 Chi‑square Test for Categorical Features

In [ ]:
categorical_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
chi2_results = []
for col in categorical_features:
    if col in df.columns:
        ct = pd.crosstab(df[col], df['target'])
        chi2, p, dof, exp = chi2_contingency(ct)
        chi2_results.append({'Feature': col, 'Chi-square': chi2, 'P-value': p})
chi2_df = pd.DataFrame(chi2_results).sort_values('P-value')
chi2_df['Significant'] = chi2_df['P-value'] < 0.05
print(chi2_df)

# 6. Feature Engineering & Preprocessing

In [ ]:
# Create BMI if height and weight exist, otherwise use existing features
if 'bmi' not in df.columns:
    # Create BMI using weight and height if available
    if 'weight' in df.columns and 'height' in df.columns:
        df['bmi'] = df['weight'] / ((df['height']/100) ** 2)

# Create age groups
df['age_group'] = pd.cut(df['age'], bins=[20,40,50,60,70,80],
                         labels=['20-40','40-50','50-60','60-70','70-80'])

In [ ]:
# Encode categorical variables
le = LabelEncoder()
for col in categorical_features:
    if col in df.columns:
        df[col] = le.fit_transform(df[col].astype(str))

In [ ]:
# Define features and target
feature_cols = [col for col in df.columns if col not in ['target', 'age_group']]
X = df[feature_cols]
y = df['target']
print(f"Features shape: {X.shape}")

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)